In [78]:
from datasets import load_dataset, DatasetDict, Dataset, Audio, disable_caching
from audio_augmentator.utils import preprocess_noise
from audio_augmentator.base_noise_dataset import get_composed_dataset
from silero_vad import load_silero_vad, get_speech_timestamps
from tqdm import tqdm
from pathlib import Path
import torchaudio
import torch
import uuid
import os

In [22]:
disable_caching()

In [ ]:
save_dataset_path = 'noises_dataset'
save_dir = ''

In [24]:
def _preprocessing(sample):
    sample['error'] = 'None'
    try:
        preprocessed_audio = preprocess_noise(sample['audio']['array'])
        if not isinstance(preprocessed_audio, torch.Tensor):
            preprocessed_audio = torch.tensor(preprocessed_audio)
        sample['preprocessed_tensor'] = preprocessed_audio
    except BaseException as e:
        sample['preprocessed_tensor'] = torch.empty((0, 0), dtype=torch.float32)
        sample['error'] = str(e)
    return sample

def preprocessing(dataset: Dataset | DatasetDict) -> DatasetDict | Dataset:
    dataset_preprocessed = dataset.map(_preprocessing).with_format("torch")
    dataset_preprocessed = dataset_preprocessed.filter(lambda example: isinstance(example['preprocessed_tensor'], torch.Tensor))
    dataset_preprocessed = dataset_preprocessed.filter(lambda example: example['preprocessed_tensor'].numel() != 0)
    return dataset_preprocessed

def save_to_files(dataset: Dataset | DatasetDict, save_prefix: str) -> DatasetDict | Dataset:

    global save_dir

    def save_sample(sample):
        new_name = str(save_dir) + '/' + save_prefix +'_' + str(uuid.uuid4()) + '.wav'
        torchaudio.save(new_name, sample['preprocessed_tensor'], 16000)

    if isinstance(dataset, DatasetDict):
        for key, value in dataset.items():
            save_dir = Path(f'{save_dataset_path}/{key}')
            save_dir.mkdir(parents=False, exist_ok=True)
            value.with_format("torch").map(save_sample)
    else:
        dataset.map(save_sample)

## Генерация сабсетов  

### audioset_nonspeech

In [ ]:
new_noises_path = "audioset_nonspeech"
new_noises_dataset = load_dataset("csv", data_files=f'{new_noises_path}/metadata.csv')['train']

In [26]:
# for rail transport or train
rail_transport_tags = [
    'rail transport', 
    'railroad', 
    'train', 
    'wagon', 
    'subway', 
    'metro', 
    'underground'
]
rail_transport_subset = new_noises_dataset.filter(
    lambda sample: any(tag in sample["label"] for tag in rail_transport_tags)
)


Filter: 100%|██████████| 13994/13994 [00:00<00:00, 552788.57 examples/s]


In [27]:
rail_transport_subset

Dataset({
    features: ['file_name', 'label'],
    num_rows: 210
})

In [28]:
# for air transport
air_transport_tags = [
    'air transport', 
    'air brake',
    'aircraft',
    'helicopter', 
    'propeller',
    'plane', 
    'jet'
]
air_transport_subset = new_noises_dataset.filter(
    lambda sample: any(tag in sample["label"] for tag in air_transport_tags) and ('music' not in sample["label"])
)

Filter: 100%|██████████| 13994/13994 [00:00<00:00, 524330.15 examples/s]


In [29]:
air_transport_subset

Dataset({
    features: ['file_name', 'label'],
    num_rows: 201
})

In [30]:
pets_tags = ['animal', 'bird', 'horse', 'snake']
pets_subset = new_noises_dataset.filter(
    lambda sample: any(tag in sample["label"] for tag in pets_tags)
)

Filter: 100%|██████████| 13994/13994 [00:00<00:00, 604636.52 examples/s]


In [31]:
pets_subset

Dataset({
    features: ['file_name', 'label'],
    num_rows: 741
})

In [32]:
def add_path(sample):
    sample['audio'] = new_noises_path + '/' + sample['audio']
    return sample

new_subsets_dict = DatasetDict()
new_subsets_dict['rail_transport'] = rail_transport_subset
new_subsets_dict['air_transport'] = air_transport_subset
new_subsets_dict['animals'] = pets_subset
new_subsets_dict = new_subsets_dict.rename_column(original_column_name='file_name', new_column_name='audio')
new_subsets_dict = new_subsets_dict.map(add_path)
new_subsets_dict = new_subsets_dict.cast_column("audio", Audio(16000))
new_subsets_dict

Map: 100%|██████████| 741/741 [00:00<00:00, 40866.50 examples/s]


DatasetDict({
    rail_transport: Dataset({
        features: ['audio', 'label'],
        num_rows: 210
    })
    air_transport: Dataset({
        features: ['audio', 'label'],
        num_rows: 201
    })
    animals: Dataset({
        features: ['audio', 'label'],
        num_rows: 741
    })
})

In [33]:
new_subsets_dict = preprocessing(new_subsets_dict)
new_subsets_dict

Filter: 100%|██████████| 735/735 [00:02<00:00, 258.75 examples/s]


DatasetDict({
    rail_transport: Dataset({
        features: ['audio', 'label', 'error', 'preprocessed_tensor'],
        num_rows: 207
    })
    air_transport: Dataset({
        features: ['audio', 'label', 'error', 'preprocessed_tensor'],
        num_rows: 199
    })
    animals: Dataset({
        features: ['audio', 'label', 'error', 'preprocessed_tensor'],
        num_rows: 735
    })
})

In [34]:
save_to_files(new_subsets_dict, 'audioset_nonspeech')

Map: 100%|██████████| 735/735 [00:03<00:00, 219.49 examples/s]


### Animal-Sound-Dataset

In [ ]:
animals_sounds_path = 'Animal-Sound-Dataset'
animals_sounds_files = [str(i) for i in list(Path(f'{animals_sounds_path}').glob('**/*.wav'))]
animals_sounds_files[0]

'/home/daniil/datasets_data/Animal-Sound-Dataset/Tavuk/tavuk_5.wav'

In [36]:
data = {"audio": animals_sounds_files}
audio_dataset = Dataset.from_dict(data)
audio_dataset = audio_dataset.cast_column("audio", Audio(16000))
audio_dataset

Dataset({
    features: ['audio'],
    num_rows: 865
})

In [37]:
audio_dataset = preprocessing(audio_dataset)

Filter: 100%|██████████| 863/863 [00:01<00:00, 436.07 examples/s]


In [38]:
save_dir = Path(f'{save_dataset_path}/animals')
save_dir.mkdir(exist_ok=True)
save_to_files(audio_dataset, 'animal_sound_dataset')

Map: 100%|██████████| 863/863 [00:02<00:00, 363.74 examples/s]


### Old subsets

In [ ]:
music_path = 'background_music'
household_path = 'household'
for old_subset in [music_path, household_path]:
    subset_path = Path(f'{old_subset}')
    old_subset_files = [str(i) for i in list(subset_path.glob('**/*.wav'))]
    old_subset_dataset = Dataset.from_dict({'audio': old_subset_files})
    old_subset_dataset = old_subset_dataset.cast_column("audio", Audio(16000))
    old_subset_dataset = preprocessing(old_subset_dataset)
    save_dir = Path(f'{save_dataset_path}/{subset_path.stem}')
    save_dir.mkdir(exist_ok=True)
    save_to_files(old_subset_dataset, subset_path.stem)

Map: 100%|██████████| 1660/1660 [00:01<00:00, 1330.77 examples/s]


## Фильтрация по голосу в шумах

In [79]:
model = load_silero_vad()
almost_final_dataset = get_composed_dataset(save_dataset_path)

def speech_checker(subset_dataset):
    speech_samples = []
    for i in tqdm(range(len(subset_dataset)), desc='Speech searching', total=len(subset_dataset)):
        speech_timestamps = get_speech_timestamps(
            subset_dataset[i],
            model,
            16000
        )
        if len(speech_timestamps) > 0:
            speech_samples.append(i)
    return speech_samples

In [80]:
not_speech_keys = [i for i in almost_final_dataset.keys() if 'speech' not in i]
for i in not_speech_keys:
    subset = almost_final_dataset[i]
    subset_speech_indices = speech_checker(subset)
    if len(subset_speech_indices) > 0:
        file_paths = [subset.file_paths[i] for i in subset_speech_indices]
        for j in file_paths:
            print(f'Удаление {j}')
            os.remove(j)

Speech searching: 100%|██████████| 5881/5881 [03:33<00:00, 27.60it/s]
